In [ ]:
import numpy as np #arrrays
import pandas as pd #data set structure
import matplotlib.pyplot as plt #visualisation data strructre
import yfinance as yf #package

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression

from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

from itertools import combinations

from sklearn.feature_selection import mutual_info_regression


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from models.lstm import PredictionModel


from features import add_features


from data_utils import create_sequences

from train import train_model

from predict import make_predictions, evaluate_predictions




[*********************100%***********************]  1 of 1 completed


In [2]:
# settings
TICKER = "AAPL"
START_DATE = "2020-01-01"

SEQ_LENGTH = 30
TRAIN_SPLIT = 0.8
HIDDEN_DIM = 64
NUM_LAYERS = 2
DROPOUT = 0.2
BATCH_SIZE = 32
LEARNING_RATE = 0.001
NUM_EPOCHS = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
df = yf.download(TICKER, start="2020-01-01")

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)
    
df = add_features(df)

[*********************100%***********************]  1 of 1 completed


In [4]:
features = ["Close", "MA10_ratio", "Return_1d", "Volatility_20"]

X = df[features]
y = df[["Tomorrow_Return"]]

In [5]:
def run_experiment(
    df, 
    TICKER, 
    SEQ_LENGTH, 
    TOP_FEATURES, 
    HIDDEN_DIM, 
    NUM_EPOCHS, 
    TRAIN_SPLIT, 
    NUM_LAYERS, 
    DROPOUT, 
    BATCH_SIZE, 
    LEARNING_RATE
):
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data = df.copy()

    # choose starting features
    features = [
        "Close",
        "Volume",
        "Return_1d",
        "Volatility_20",
        "MA10_ratio",
        "HL_Range",
        "Volume_Ratio"
    ]

    X = data[features]
    y = data["Tomorrow_Return"]

    # train/test split
    train_size = int(TRAIN_SPLIT * len(data))

    X_train_raw = X.iloc[:train_size]
    X_test_raw = X.iloc[train_size:]

    y_train_raw = y.iloc[:train_size]
    y_test_raw = y.iloc[train_size:]

    # mutual information feature selection
    mi = mutual_info_regression(X_train_raw, y_train_raw, random_state=42)

    mi_scores = pd.Series(mi, index=X_train_raw.columns)
    mi_scores = mi_scores.sort_values(ascending=False)

    top_features = mi_scores.head(TOP_FEATURES).index.tolist()

    X_train_raw = X_train_raw[top_features]
    X_test_raw = X_test_raw[top_features]

    # scaling
    feature_scaler = MinMaxScaler()
    target_scaler = MinMaxScaler()

    X_train_scaled = feature_scaler.fit_transform(X_train_raw)
    X_test_scaled = feature_scaler.transform(X_test_raw)

    y_train_scaled = target_scaler.fit_transform(y_train_raw.to_frame())
    y_test_scaled = target_scaler.transform(y_test_raw.to_frame())

    # sequences
    X_train_tensor, y_train_tensor = create_sequences(
        X_train_scaled, 
        y_train_scaled, 
        SEQ_LENGTH
    )

    X_test_tensor, y_test_tensor = create_sequences(
        X_test_scaled, 
        y_test_scaled, 
        SEQ_LENGTH
    )

    # dataloader
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False
    )

    # model
    model = PredictionModel(
        input_dim=len(top_features),
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        output_dim=1,
        dropout=DROPOUT
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # train
    model = train_model(
        model=model,
        train_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        num_epochs=NUM_EPOCHS
    )

    y_train_pred, y_test_pred, y_train_actual, y_test_actual = make_predictions(
        model=model,
        X_train=X_train_tensor,
        X_test=X_test_tensor,
        y_train=y_train_tensor,
        y_test=y_test_tensor,
        target_scaler=target_scaler,
        device=device
    )

    train_rmse, test_rmse, direction = evaluate_predictions(
        y_train_pred,
        y_test_pred,
        y_train_actual,
        y_test_actual
    )

    return train_rmse, test_rmse, direction, y_test_pred, y_test_actual
    


In [ ]:
    results = []
    
    for seq_length in [5, 10, 20, 30, 60]:
        train_rmse, test_rmse, direction, y_test_pred, y_test = run_experiment(df, TICKER, SEQ_LENGTH = seq_length, TOP_FEATURES=10, HIDDEN_DIM = 64, NUM_EPOCHS = 100, TRAIN_SPLIT = 0.8, NUM_LAYERS = 2, DROPOUT = 0.2 , BATCH_SIZE = 32, LEARNING_RATE = 0.001
        )
    
        results.append({
            "Sequence Length": seq_length,
            "Train RMSE": train_rmse,
            "Test RMSE": test_rmse,
            "Direction": direction,
            "Mean return prediction:": np.mean(y_test_pred),
            "Mean actual return:": np.mean(y_test),
            "Model std": np.std(y_test_pred),
            "Actual std": np.std(y_test)
        })
        print(results)
    
    results = pd.DataFrame(results)
    print(results)

Epoch 0, Loss: 0.002584
Epoch 10, Loss: 0.001857
Epoch 20, Loss: 0.001492
Epoch 30, Loss: 0.001612
Epoch 40, Loss: 0.001748
Epoch 50, Loss: 0.001735
Epoch 60, Loss: 0.001612
Epoch 70, Loss: 0.001655
Epoch 80, Loss: 0.001647
Epoch 90, Loss: 0.001377
[{'Sequence Length': 5, 'Train RMSE': 0.0206947922706604, 'Test RMSE': 0.01947684958577156, 'Direction': np.float64(0.5440251572327044), 'Mean return prediction:': np.float32(0.002544337), 'Mean actual return:': np.float32(0.0012263262), 'Model std': np.float32(0.0023742875), 'Actual std': np.float32(0.019485274)}]
Epoch 0, Loss: 0.007785
Epoch 10, Loss: 0.004570
Epoch 20, Loss: 0.004888
Epoch 30, Loss: 0.004898
Epoch 40, Loss: 0.005013
Epoch 50, Loss: 0.004837
Epoch 60, Loss: 0.004978
Epoch 70, Loss: 0.004998
Epoch 80, Loss: 0.004931
Epoch 90, Loss: 0.005034
[{'Sequence Length': 5, 'Train RMSE': 0.0206947922706604, 'Test RMSE': 0.01947684958577156, 'Direction': np.float64(0.5440251572327044), 'Mean return prediction:': np.float32(0.00254433